# Spark Exercise

Apache Spark is an excellent tool for data engineering projects due to its robust ability to process large-scale data efficiently through distributed computing. Spark's in-memory processing capabilities significantly enhance the speed of data operations, making it ideal for handling big data workloads. It supports various data sources and formats, offering versatility in data ingestion and transformation. Additionally, Spark's rich API supports multiple programming languages such as Python, Java, and Scala, catering to diverse developer preferences. Its ecosystem, which includes libraries for SQL, machine learning, and graph processing, provides a comprehensive suite for building complex data pipelines and analytics, making it a powerful and flexible choice for data engineering tasks.

Use Python, ```pyspark``` and ```pandas``` to explore Apache Spark RDD and DataFrame:

# Spark RDD

Spark RDD (Resilient Distributed Dataset) is a fundamental data structure in Apache Spark that enables fault-tolerant, distributed processing of large datasets across multiple nodes in a cluster. Spark RDDs provide a higher-level abstraction for performing distributed data processing tasks, including both map (transformations) and reduce (aggregations) operations.

## Import Necessary Libraries

In [20]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd

spark = SparkSession.builder \
    .appName("ViennaMarathonSpark") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

print("SparkContext:", sc)
print("SparkSession:", spark)

SparkContext: <SparkContext master=local[*] appName=ViennaMarathonSpark>
SparkSession: <pyspark.sql.session.SparkSession object at 0x7b451c6cb290>


## Spark Context and Session
Initialize Spark Context and Spark Session

In [3]:
# siehe vorherige Code-Zelle (hier eine neue Session zu erstellen führt zu Fehlermeldungen)

## Load Data into RDD

In [21]:
raw_rdd = sc.textFile("vienna_city_marathon_all_years_participants.csv")

header = raw_rdd.first()
data_rdd = raw_rdd.filter(lambda line: line != header and line.strip() != "")

print("Header:", header)
print("Total data rows:", data_rdd.count())

Header: year,event_name,bib_number,participant_id,run_time
Total data rows: 15000


## Map Operation

Split data into individual parts and create key-value pairs

In [23]:
def time_to_seconds(t):
    """Convert HH:MM:SS string to total seconds."""
    try:
        h, m, s = t.strip().split(":")
        return int(h) * 3600 + int(m) * 60 + int(s)
    except Exception:
        return None

def parse_row(line):
    parts = line.split(",")
    try:
        year = int(parts[0].strip())
        run_time = parts[4].strip() if len(parts) > 4 else None
        seconds = time_to_seconds(run_time) if run_time else None
        if seconds:
            return (year, seconds)
    except Exception:
        pass
    return None

mapped_rdd = data_rdd.map(parse_row).filter(lambda x: x is not None)

print("Mapped key-value pairs (year, run_time_seconds):")

Mapped key-value pairs (year, run_time_seconds):


## Reduce Operation

Reduce your key-value pairs

In [24]:
sum_count_rdd = mapped_rdd.map(lambda kv: (kv[0], (kv[1], 1)))
reduced_rdd = sum_count_rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

def seconds_to_time(s):
    s = int(s)
    return f"{s // 3600:02d}:{(s % 3600) // 60:02d}:{s % 60:02d}"

avg_time_rdd = reduced_rdd.map(
    lambda kv: (kv[0], seconds_to_time(kv[1][0] / kv[1][1]), kv[1][1])
)


## Collect Results

Because of lazy evaluation, the map-reduce operation is performed only now. Show what you calculated.

In [25]:
results = avg_time_rdd.sortBy(lambda x: x[0]).collect()

## Save Results

In [26]:
import os
output_path = os.path.join(os.getcwd(), "rdd_marathon_avg_times.csv")
results_df = pd.DataFrame(results, columns=["year", "avg_finish_time", "num_finishers"])
results_df.to_csv(output_path, index=False)
print("Saved to:", output_path)

Saved to: /home/jovyan/rdd_marathon_avg_times.csv


# Spark DataFrame

Spark DataFrame is a distributed collection of data organized into named columns, designed for efficient data manipulation and analysis in Apache Spark. It is used for various data processing tasks such as data ingestion, transformation, querying, and analysis in Apache Spark, providing a high-level abstraction that simplifies working with structured data.

## Load Data into DataFrame

In [27]:
pandas_df = pd.read_csv("vienna_city_marathon_all_years_participants.csv")

df = spark.createDataFrame(pandas_df)
print(f"Loaded DataFrame: {df.count()} rows, {len(df.columns)} columns")

Loaded DataFrame: 15000 rows, 5 columns


## View DataFrame Schema

In [28]:
df.printSchema()

root
 |-- year: long (nullable = true)
 |-- event_name: string (nullable = true)
 |-- bib_number: double (nullable = true)
 |-- participant_id: string (nullable = true)
 |-- run_time: string (nullable = true)



## View DataFrame Data

In [29]:
df.show(10, truncate=False)

+----+--------------------+----------+--------------+--------+
|year|event_name          |bib_number|participant_id|run_time|
+----+--------------------+----------+--------------+--------+
|2024|Vienna City Marathon|NaN       |HCH8OHST1B9C1D|03:31:48|
|2024|Vienna City Marathon|NaN       |HCH8OHST1C8B01|02:06:35|
|2024|Vienna City Marathon|NaN       |HCH8OHST1BA7DD|03:05:48|
|2024|Vienna City Marathon|NaN       |HCH8OHST1C8B0D|02:24:08|
|2024|Vienna City Marathon|NaN       |HCH8OHST1BA587|03:31:46|
|2024|Vienna City Marathon|NaN       |HCH8OHST1C8B07|02:10:42|
|2024|Vienna City Marathon|NaN       |HCH8OHST1BA716|03:05:27|
|2024|Vienna City Marathon|NaN       |HCH8OHST1C8B25|02:26:22|
|2024|Vienna City Marathon|NaN       |HCH8OHST1BB69F|03:44:04|
|2024|Vienna City Marathon|NaN       |HCH8OHST1C8B26|02:10:44|
+----+--------------------+----------+--------------+--------+
only showing top 10 rows



## Filter Data

Performe a filter operation on a column

In [40]:
df.createOrReplaceTempView("marathon")

finishers = spark.sql("""
    SELECT * FROM marathon
    WHERE run_time IS NOT NULL
""")

print(f"Total participants: {df.count()}")
print(f"Finishers (with run_time): {finishers.count()}")
print(f"missing time: {df.count() - finishers.count()}")
finishers.show(10, truncate=False)

Total participants: 15000
Finishers (with run_time): 15000
missing time: 0
+----+--------------------+----------+--------------+--------+
|year|event_name          |bib_number|participant_id|run_time|
+----+--------------------+----------+--------------+--------+
|2024|Vienna City Marathon|NaN       |HCH8OHST1B9C1D|03:31:48|
|2024|Vienna City Marathon|NaN       |HCH8OHST1C8B01|02:06:35|
|2024|Vienna City Marathon|NaN       |HCH8OHST1BA7DD|03:05:48|
|2024|Vienna City Marathon|NaN       |HCH8OHST1C8B0D|02:24:08|
|2024|Vienna City Marathon|NaN       |HCH8OHST1BA587|03:31:46|
|2024|Vienna City Marathon|NaN       |HCH8OHST1C8B07|02:10:42|
|2024|Vienna City Marathon|NaN       |HCH8OHST1BA716|03:05:27|
|2024|Vienna City Marathon|NaN       |HCH8OHST1C8B25|02:26:22|
|2024|Vienna City Marathon|NaN       |HCH8OHST1BB69F|03:44:04|
|2024|Vienna City Marathon|NaN       |HCH8OHST1C8B26|02:10:44|
+----+--------------------+----------+--------------+--------+
only showing top 10 rows



## Group By and Aggregate

Performe a group by and aggregat operation

In [50]:
yearly_stats = spark.sql("""
    SELECT
        year,
        COUNT(*) AS total_participants,
        SUM(CASE WHEN run_time < '03:00:00' THEN 1 ELSE 0 END) AS under_3h,
        SUM(CASE WHEN run_time < '02:30:00' THEN 1 ELSE 0 END) AS under_2h30,
        MIN(CASE WHEN run_time IS NOT NULL AND run_time != '' AND run_time != 'NaN' THEN run_time END) AS fastest,
        MAX(CASE WHEN run_time IS NOT NULL AND run_time != '' AND run_time != 'NaN' THEN run_time END) AS slowest
    FROM marathon
    GROUP BY year
    ORDER BY year
""")

yearly_stats.show()

+----+------------------+--------+----------+--------+--------+
|year|total_participants|under_3h|under_2h30| fastest| slowest|
+----+------------------+--------+----------+--------+--------+
|2024|              5000|     500|        43|02:06:35|06:26:59|
|2025|              5000|     578|        27|02:08:28|06:09:40|
|2026|              5000|     433|        25|02:06:53|06:56:38|
+----+------------------+--------+----------+--------+--------+



## Save DataFrame to Parquet

In [51]:
yearly_stats.write.mode("overwrite").parquet("marathon_yearly_stats.parquet")
verify_df = spark.read.parquet("marathon_yearly_stats.parquet")
verify_df.show()

+----+------------------+--------+----------+--------+--------+
|year|total_participants|under_3h|under_2h30| fastest| slowest|
+----+------------------+--------+----------+--------+--------+
|2024|              5000|     500|        43|02:06:35|06:26:59|
|2025|              5000|     578|        27|02:08:28|06:09:40|
|2026|              5000|     433|        25|02:06:53|06:56:38|
+----+------------------+--------+----------+--------+--------+

